In [3]:
import pandas as pd

# TABLE CLIENTS
## Workflow Bronze / Silver / Gold

In [4]:
# Bronze : donnees brutes
clients_bronze = pd.read_csv('data/raw/client.csv')
clients_bronze.head()

,id,nom,email,city,pays,date_inscription
0,1,Marc Lacroix-Seguin,marc.seguin5862@shopflow.com,La Plata,Argentine,2025-08-13
1,2,Roger Perrot,roger.perrot9726@shopflow.com,Los Angeles,Etats-Unis,2026-02-27
2,3,Simone Marin du Clément,simone.clement8102@shopflow.com,Kyoto,Japon,2025-08-19
3,4,Léon Imbert,leon.imbert2512@shopflow.com,Buenos Aires,Argentine,2025-08-13
4,5,Bernard Lopez-Perrot,bernard.perrot3663@shopflow.com,Tanger,Maroc,2025-06-13


In [5]:
# 1) Regles de qualite simples
pays_autorises = [
    'Australie', 'Japon', 'Egypte', 'Nigeria', 'Allemagne',
    'France', 'Canada', 'Etats-Unis', 'Bresil', 'Argentine',
    'Maroc', 'Afrique du Sud', 'Kenya', 'Chine', 'Inde',
    'Coree du Sud', 'Indonesie', 'Nouvelle-Zelande', 'Mexique', 'Turquie'
 ]

print('Lignes bronze:', len(clients_bronze))
print('ID uniques:', clients_bronze['id'].is_unique)
print('Pays autorises:', clients_bronze['pays'].isin(pays_autorises).sum(), '/', len(clients_bronze))
clients_bronze.info()
clients_bronze.describe()
clients_bronze['pays'].value_counts()

Lignes bronze: 200
ID uniques: True
Pays autorises: 200 / 200
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                200 non-null    int64
 1   nom               200 non-null    str  
 2   email             194 non-null    str  
 3   city              190 non-null    str  
 4   pays              200 non-null    str  
 5   date_inscription  200 non-null    str  
dtypes: int64(1), str(5)
memory usage: 23.7 KB


pays
Turquie             17
Bresil              15
Australie           15
Chine               12
Kenya               11
Afrique du Sud      11
Coree du Sud        11
Maroc               10
Egypte              10
Nouvelle-Zelande    10
France               9
Nigeria              9
Argentine            8
Indonesie            8
Inde                 8
Mexique              8
Allemagne            8
Etats-Unis           7
Japon                7
Canada               6
Name: count, dtype: int64

In [6]:
# Silver : nettoyage simple
clients_silver = clients_bronze.copy()

clients_silver['nom'] = clients_silver['nom'].astype(str).str.strip()
clients_silver['email'] = clients_silver['email'].astype(str).str.strip().str.lower()
clients_silver['city'] = clients_silver['city'].astype(str).str.strip()
clients_silver['pays'] = clients_silver['pays'].astype(str).str.strip()

clients_silver['date_inscription'] = pd.to_datetime(
    clients_silver['date_inscription'],
    errors='coerce'
 )
clients_silver.isna().sum()


id                   0
nom                  0
email                6
city                10
pays                 0
date_inscription     0
dtype: int64

In [7]:
clients_silver[clients_silver['city'].isna()]

,id,nom,email,city,pays,date_inscription
19,20,Laurent Maillot,laurent.maillot4683@@shopflow.com,NaN,Maroc,2025-12-04
39,40,Théodore Descamps,theodore.descamps6757@,NaN,Inde,2025-12-18
59,60,Anastasie Tanguy,@shopflow.com,NaN,Turquie,2025-09-23
79,80,Élisabeth Salmon,elisabeth.salmon860.shopflow.com,NaN,France,2025-12-30
99,100,Alfred Perret Le Leblanc,alfred.leblanc288@,NaN,Kenya,2025-12-12
119,120,Grégoire Huet,gregoire.huet1537@,NaN,Chine,2025-05-05
139,140,Étienne Godard,etienne.godard2314.shopflow.com,NaN,Turquie,2025-12-09
159,160,Élise Fabre,elise.fabre6599.shopflow.com,NaN,Indonesie,2025-09-08
179,180,Roger Guillou,roger.guillou1351.shopflow.com,NaN,Nouvelle-Zelande,2026-02-15
199,200,Constance Ollivier,@shopflow.com,NaN,Bresil,2025-10-15


In [8]:
clients_silver[clients_silver['email'].isna()]


,id,nom,email,city,pays,date_inscription
32,33,Jules Simon,NaN,Puebla,Mexique,2025-11-05
65,66,Madeleine Pasquier,NaN,Calgary,Canada,2025-08-31
98,99,Sophie-Adélaïde Pons,NaN,Dunedin,Nouvelle-Zelande,2026-01-13
131,132,Emmanuel Vasseur,NaN,Berlin,Allemagne,2026-01-21
164,165,Richard-Honoré Gilles,NaN,Sydney,Australie,2026-01-03
197,198,Nicole Delaunay,NaN,Cologne,Allemagne,2025-11-15


In [9]:
# Flags qualité
clients_silver['email_missing'] = clients_silver['email'].isna()
clients_silver['city_missing'] = clients_silver['city'].isna()

# Quarantine : lignes rejetées (email manquant)
clients_quarantine = clients_silver[clients_silver['email_missing']].copy()
clients_quarantine

,id,nom,email,city,pays,date_inscription,email_missing,city_missing
32,33,Jules Simon,NaN,Puebla,Mexique,2025-11-05,True,False
65,66,Madeleine Pasquier,NaN,Calgary,Canada,2025-08-31,True,False
98,99,Sophie-Adélaïde Pons,NaN,Dunedin,Nouvelle-Zelande,2026-01-13,True,False
131,132,Emmanuel Vasseur,NaN,Berlin,Allemagne,2026-01-21,True,False
164,165,Richard-Honoré Gilles,NaN,Sydney,Australie,2026-01-03,True,False
197,198,Nicole Delaunay,NaN,Cologne,Allemagne,2025-11-15,True,False


In [10]:
clients_silver['city']=clients_silver['city'].fillna('inconnue')


In [11]:
clients_silver.isna().sum()

id                  0
nom                 0
email               6
city                0
pays                0
date_inscription    0
email_missing       0
city_missing        0
dtype: int64

In [12]:
clients_silver.duplicated().sum()

np.int64(0)

In [13]:
# On garde seulement les lignes valides
clients_silver = clients_silver[clients_silver['email'].str.contains('@', na=False)]
clients_silver = clients_silver[clients_silver['pays'].isin(pays_autorises)]

# On supprime les doublons sur id
clients_silver = clients_silver.drop_duplicates(subset=['id'], keep='first')

print('Lignes silver:', len(clients_silver))
print('Doublons restants:', clients_silver['id'].duplicated().sum())
print('Emails invalides restants:', (~clients_silver['email'].str.contains('@', na=False)).sum())

Lignes silver: 188
Doublons restants: 0
Emails invalides restants: 0


In [14]:
# Gold : table propre et exploitable
# Nettoyage des colonnes techniques pour relancer la cellule sans erreur
clients_silver = clients_silver.drop(columns=['client_id'], errors='ignore')
clients_silver = clients_silver.drop(columns=['client_id_x'], errors='ignore')
clients_silver = clients_silver.drop(columns=['client_id_y'], errors='ignore')
clients_silver = clients_silver.drop(columns=['nb_commandes'], errors='ignore')
clients_silver = clients_silver.drop(columns=['categorie_client'], errors='ignore')
clients_silver = clients_silver.drop(columns=['nb_commandes_x'], errors='ignore')
clients_silver = clients_silver.drop(columns=['nb_commandes_y'], errors='ignore')
clients_silver = clients_silver.drop(columns=['email_missing'], errors='ignore')
clients_silver = clients_silver.drop(columns=['city_missing'], errors='ignore')

# Source autonome: calcule le nb de commandes client depuis la table brute
commande_base = pd.read_csv('data/raw/commande.csv')
nb_commandes_tmp = commande_base.groupby('client_id').size().reset_index(name='nb_commandes_tmp')
clients_silver = clients_silver.merge(nb_commandes_tmp, left_on='id', right_on='client_id', how='left')
clients_silver['nb_commandes_tmp'] = clients_silver['nb_commandes_tmp'].fillna(0).astype('Int64')
clients_silver['categorie_client'] = clients_silver['nb_commandes_tmp'].apply(lambda x: 'fidele' if x > 5 else 'nouveau')
clients_silver = clients_silver.drop(columns=['client_id', 'nb_commandes_tmp'], errors='ignore')

# Arrondi des colonnes monetaires a 2 decimales
if 'revenue_eur' in clients_silver.columns:
    clients_silver['revenue_eur'] = clients_silver['revenue_eur'].round(2)
if 'revenue_usd' in clients_silver.columns:
    clients_silver['revenue_usd'] = clients_silver['revenue_usd'].round(2)
if 'total_eur' in clients_silver.columns:
    clients_silver['total_eur'] = clients_silver['total_eur'].round(2)
if 'total_usd' in clients_silver.columns:
    clients_silver['total_usd'] = clients_silver['total_usd'].round(2)

clients_gold = clients_silver.copy()
clients_gold.isna().sum()
print('Lignes gold:', len(clients_gold))
print('Colonnes gold:', list(clients_gold.columns))
clients_gold.head(12)


Lignes gold: 188
Colonnes gold: ['id', 'nom', 'email', 'city', 'pays', 'date_inscription', 'categorie_client']


,id,nom,email,city,pays,date_inscription,categorie_client
0,1,Marc Lacroix-Seguin,marc.seguin5862@shopflow.com,La Plata,Argentine,2025-08-13,nouveau
1,2,Roger Perrot,roger.perrot9726@shopflow.com,Los Angeles,Etats-Unis,2026-02-27,fidele
2,3,Simone Marin du Clément,simone.clement8102@shopflow.com,Kyoto,Japon,2025-08-19,fidele
3,4,Léon Imbert,leon.imbert2512@shopflow.com,Buenos Aires,Argentine,2025-08-13,fidele
4,5,Bernard Lopez-Perrot,bernard.perrot3663@shopflow.com,Tanger,Maroc,2025-06-13,fidele
5,6,Paulette Becker,paulette.becker196@shopflow.com,Assouan,Egypte,2026-03-29,nouveau
6,7,Alexandre de Petitjean,alexandre.petitjean6277@shopflow.com,Kisumu,Kenya,2025-09-24,fidele
7,8,Constance Mathieu,constance.mathieu4246@shopflow.com,Yogyakarta,Indonesie,2026-04-03,nouveau
8,9,Laetitia de Hubert,laetitia.hubert638@shopflow.com,Kyoto,Japon,2025-06-26,nouveau
9,10,Inès Ruiz,ines.ruiz1209@,Christchurch,Nouvelle-Zelande,2025-12-29,fidele


In [15]:
# Export final de la table Gold
clients_gold.to_csv('data/clean/client_clean.csv', index=False)

print('Fichier cree: data/clean/client_clean.csv')
clients_gold.head()

Fichier cree: data/clean/client_clean.csv


,id,nom,email,city,pays,date_inscription,categorie_client
0,1,Marc Lacroix-Seguin,marc.seguin5862@shopflow.com,La Plata,Argentine,2025-08-13,nouveau
1,2,Roger Perrot,roger.perrot9726@shopflow.com,Los Angeles,Etats-Unis,2026-02-27,fidele
2,3,Simone Marin du Clément,simone.clement8102@shopflow.com,Kyoto,Japon,2025-08-19,fidele
3,4,Léon Imbert,leon.imbert2512@shopflow.com,Buenos Aires,Argentine,2025-08-13,fidele
4,5,Bernard Lopez-Perrot,bernard.perrot3663@shopflow.com,Tanger,Maroc,2025-06-13,fidele


# TABLE Commandes
## Workflow Bronze / Silver / Gold

In [16]:
commande_bronze=pd.read_csv('data/raw/commande.csv')

In [17]:
commande_bronze.head()

,id,client_id,produit_id,quantité,total_eur,status,date_commande
0,1,179,47,5.0,2.211942e+06,pending,2025-12-23
1,2,3,18,1.0,2.268010e+05,completed,2025-10-08
2,3,20,39,2.0,2.143343e+06,refunded,2025-09-22
3,4,16,38,3.0,3.806314e+05,completed,2025-07-05
4,5,95,37,5.0,2.249981e+06,completed,2025-08-06


In [18]:
# 1)Regles de qualite simples

print('Lignes bronze:', len(commande_bronze))
print('ID uniques:', commande_bronze['id'].is_unique)
commande_bronze.info()
commande_bronze.describe()

Lignes bronze: 1015
ID uniques: True
<class 'pandas.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             1015 non-null   int64  
 1   client_id      1015 non-null   int64  
 2   produit_id     1015 non-null   int64  
 3   quantité       995 non-null    float64
 4   total_eur      1015 non-null   float64
 5   status         1015 non-null   str    
 6   date_commande  1015 non-null   str    
dtypes: float64(2), int64(3), str(2)
memory usage: 74.1 KB


,id,client_id,produit_id,quantité,total_eur
count,1015.000000,1015.000000,1015.000000,995.000000,1.015000e+03
mean,508.000000,99.100493,26.026601,3.064322,1.553718e+06
std,293.149564,58.965070,14.133705,1.412036,8.671723e+05
min,1.000000,1.000000,1.000000,1.000000,3.655724e+03
25%,254.500000,46.000000,14.000000,2.000000,7.986622e+05
50%,508.000000,98.000000,26.000000,3.000000,1.620366e+06
75%,761.500000,150.500000,38.000000,4.000000,2.297442e+06
max,1015.000000,200.000000,50.000000,5.000000,2.999720e+06


In [19]:
# Silver : nettoyage simple
commande_silver = commande_bronze.copy()
commande_silver['quantité'] = pd.to_numeric(commande_silver['quantité'], errors='coerce')
commande_silver['date_commande'] = pd.to_datetime(
    commande_silver['date_commande'],
    errors='coerce'
 )
print('valeurs nuls',commande_silver.isna().sum())
print('----------------------')
print('nmbre de doublons:',commande_silver.duplicated(
    subset=['client_id', 'produit_id', 'quantité', 'date_commande', 'status']
).sum())
print('----------------------')
print('nb_invalide:',(commande_silver['quantité'] <= 0).sum())



valeurs nuls id                0
client_id         0
produit_id        0
quantité         20
total_eur         0
status            0
date_commande     0
dtype: int64
----------------------
nmbre de doublons: 15
----------------------
nb_invalide: 0


In [20]:
commande_silver[commande_silver['quantité'].isna()]


,id,client_id,produit_id,quantité,total_eur,status,date_commande
49,50,107,38,NaN,2.408671e+06,refunded,2025-09-27
99,100,164,27,NaN,4.147849e+05,completed,2025-07-31
149,150,101,30,NaN,2.935505e+06,completed,2025-12-12
199,200,99,26,NaN,2.851642e+06,pending,2025-12-11
249,250,3,40,NaN,1.872388e+06,completed,2025-08-20
299,300,33,8,NaN,1.702303e+06,completed,2025-08-06
349,350,175,10,NaN,7.209738e+05,completed,2025-10-20
399,400,69,25,NaN,2.118850e+06,pending,2025-07-11
449,450,13,3,NaN,2.841434e+06,cancelled,2025-08-10
499,500,34,29,NaN,1.736426e+06,completed,2025-07-26


In [21]:

# Regle metier: une quantite <= 0 est consideree invalide
commande_silver.loc[commande_silver['quantité'] <= 0, 'quantité'] = pd.NA

# Imputation: mediane par produit, puis fallback a 1 si aucune mediane disponible
mediane_par_produit = commande_silver.groupby('produit_id')['quantité'].transform('median')
commande_silver['quantité'] = (
    commande_silver['quantité']
    .fillna(mediane_par_produit)
    .fillna(1)
    .round()
    .astype('Int64')
)

nb_null_apres = commande_silver['quantité'].isna().sum()

print(f"Quantites nulles apres: {nb_null_apres}")

Quantites nulles apres: 0


In [22]:
commande_silver.loc[49:100, 'id':'quantité']

,id,client_id,produit_id,quantité
49,50,107,38,3
50,51,75,12,5
51,52,66,48,5
52,53,64,1,5
53,54,45,20,1
54,55,157,37,5
55,56,157,19,3
56,57,166,15,4
57,58,95,40,3
58,59,15,19,3


In [23]:
commande_silver[commande_silver.duplicated(
    subset=['client_id', 'produit_id', 'quantité', 'date_commande', 'status'],
    keep=False
)].sort_values(by='client_id')

,id,client_id,produit_id,quantité,total_eur,status,date_commande
704,705,17,14,5,6.286616e+05,completed,2025-11-12
1004,1005,17,14,5,6.286616e+05,completed,2025-11-12
1008,1009,45,15,5,2.366188e+05,completed,2025-09-27
985,986,45,15,5,2.366188e+05,completed,2025-09-27
78,79,65,4,1,2.491717e+06,cancelled,2025-10-08
1010,1011,65,4,1,2.491717e+06,cancelled,2025-10-08
1002,1003,74,34,4,2.736062e+06,refunded,2025-11-15
804,805,74,34,4,2.736062e+06,refunded,2025-11-15
601,602,78,28,5,7.046866e+05,completed,2025-12-10
1009,1010,78,28,5,7.046866e+05,completed,2025-12-10


In [24]:
commande_silver = commande_silver.drop_duplicates()

In [25]:
# Gold : table propre et exploitable
taux_change_usd = 1.08

# Suppression des colonnes mois si elles existent
commande_silver = commande_silver.drop(columns=['mois_commande'], errors='ignore')
commande_silver = commande_silver.drop(columns=['mois_commandes'], errors='ignore')

commande_silver['revenue_eur'] = commande_silver['total_eur']
commande_silver['revenue_usd'] = commande_silver['revenue_eur'] * taux_change_usd

# Arrondi simple des montants a 2 décimales
commande_silver['total_eur'] = commande_silver['total_eur'].round(2)
commande_silver['revenue_eur'] = commande_silver['revenue_eur'].round(2)
commande_silver['revenue_usd'] = commande_silver['revenue_usd'].round(2)
if 'total_usd' in commande_silver.columns:
    commande_silver['total_usd'] = commande_silver['total_usd'].round(2)

commande_gold = commande_silver.copy()
commande_gold.isna().sum()
commande_silver.duplicated().sum()
print('Lignes gold:', len(commande_gold))
print('Colonnes gold:', list(commande_gold.columns))
commande_gold.head()


Lignes gold: 1015
Colonnes gold: ['id', 'client_id', 'produit_id', 'quantité', 'total_eur', 'status', 'date_commande', 'revenue_eur', 'revenue_usd']


,id,client_id,produit_id,quantité,total_eur,status,date_commande,revenue_eur,revenue_usd
0,1,179,47,5,2211941.59,pending,2025-12-23,2211941.59,2388896.92
1,2,3,18,1,226800.98,completed,2025-10-08,226800.98,244945.06
2,3,20,39,2,2143342.73,refunded,2025-09-22,2143342.73,2314810.15
3,4,16,38,3,380631.45,completed,2025-07-05,380631.45,411081.97
4,5,95,37,5,2249981.41,completed,2025-08-06,2249981.41,2429979.92


In [26]:
# Export final de la table Gold
commande_gold.to_csv('data/clean/commande_clean.csv', index=False)

print('Fichier cree: data/clean/commande_clean.csv')
commande_gold.head()

Fichier cree: data/clean/commande_clean.csv


,id,client_id,produit_id,quantité,total_eur,status,date_commande,revenue_eur,revenue_usd
0,1,179,47,5,2211941.59,pending,2025-12-23,2211941.59,2388896.92
1,2,3,18,1,226800.98,completed,2025-10-08,226800.98,244945.06
2,3,20,39,2,2143342.73,refunded,2025-09-22,2143342.73,2314810.15
3,4,16,38,3,380631.45,completed,2025-07-05,380631.45,411081.97
4,5,95,37,5,2249981.41,completed,2025-08-06,2249981.41,2429979.92


# TABLE Produit
## Workflow Bronze / Silver / Gold

In [27]:
# Bronze : charger le fichier brut
produit_bronze = pd.read_csv('data/raw/produit.csv')

print('Lignes bronze:', len(produit_bronze))
print('Colonnes:', list(produit_bronze.columns))
print('Doublons id:', produit_bronze['id'].duplicated().sum())
print('Valeurs nulles:')
print(produit_bronze.isna().sum())
produit_bronze.head()

Lignes bronze: 50
Colonnes: ['id', 'categorie', 'name', 'prix_eur', 'stock']
Doublons id: 0
Valeurs nulles:
id           0
categorie    0
name         0
prix_eur     0
stock        0
dtype: int64


,id,categorie,name,prix_eur,stock
0,1,Clothing,Basic T-Shirt,379.124887,229
1,2,Home,Storage Box,345.565172,134
2,3,Electronics,Bluetooth Speaker,373.421185,149
3,4,Sports,Fitness Band,291.706869,336
4,5,Sports,Water Bottle,47.915733,412


In [28]:
# Silver : nettoyage simple
produit_silver = produit_bronze.copy()

# Nettoyer les espaces inutiles
produit_silver['name'] = produit_silver['name'].astype(str).str.strip()
produit_silver['categorie'] = produit_silver['categorie'].astype(str).str.strip()

# Convertir en types appropriés
produit_silver['prix_eur'] = pd.to_numeric(produit_silver['prix_eur'], errors='coerce')
produit_silver['stock'] = pd.to_numeric(produit_silver['stock'], errors='coerce').astype('Int64')

# Vérifier les doublons
print('Valeurs nulles:', produit_silver.isna().sum().sum())
print('Doublons sur id:', produit_silver['id'].duplicated().sum())
print('Lignes silver:', len(produit_silver))
produit_silver.head()

Valeurs nulles: 0
Doublons sur id: 0
Lignes silver: 50


,id,categorie,name,prix_eur,stock
0,1,Clothing,Basic T-Shirt,379.124887,229
1,2,Home,Storage Box,345.565172,134
2,3,Electronics,Bluetooth Speaker,373.421185,149
3,4,Sports,Fitness Band,291.706869,336
4,5,Sports,Water Bottle,47.915733,412


In [29]:
# Gold : table finale
produit_silver = produit_silver.drop(columns=['prix_usd'], errors='ignore')

produit_silver['prix_eur'] = produit_silver['prix_eur'].round(2)

taux_change = 1.08
produit_silver['prix_usd'] = (produit_silver['prix_eur'] * taux_change).round(2)

produit_gold = produit_silver.copy()

print('Lignes gold:', len(produit_gold))
print('Colonnes gold:', list(produit_gold.columns))
produit_gold.head()

Lignes gold: 50
Colonnes gold: ['id', 'categorie', 'name', 'prix_eur', 'stock', 'prix_usd']


,id,categorie,name,prix_eur,stock,prix_usd
0,1,Clothing,Basic T-Shirt,379.12,229,409.45
1,2,Home,Storage Box,345.57,134,373.22
2,3,Electronics,Bluetooth Speaker,373.42,149,403.29
3,4,Sports,Fitness Band,291.71,336,315.05
4,5,Sports,Water Bottle,47.92,412,51.75


In [30]:
# Export final de la table Gold
produit_gold.to_csv('data/clean/produit_clean.csv', index=False)

print('Fichier cree: data/clean/produit_clean.csv')
produit_gold.head()

Fichier cree: data/clean/produit_clean.csv


,id,categorie,name,prix_eur,stock,prix_usd
0,1,Clothing,Basic T-Shirt,379.12,229,409.45
1,2,Home,Storage Box,345.57,134,373.22
2,3,Electronics,Bluetooth Speaker,373.42,149,403.29
3,4,Sports,Fitness Band,291.71,336,315.05
4,5,Sports,Water Bottle,47.92,412,51.75
